# Klasifikasi Aksara Tradisional Nusantara — INFEST XII 2026

Notebook ini berisi seluruh proses dari data mentah sampai berkas submission dalam satu
berkas, tanpa memuat script, checkpoint, maupun artifact dari luar. Satu-satunya masukan
eksternal adalah bobot pretrained publik `vit_large_patch16_dinov3.lvd1689m` dari timm,
yang diperbolehkan Rule 5 panitia sebagai base model untuk fine-tuning.

## Ringkasan pendekatan

Model dilatih pada **seluruh** data train, lalu prediksi test dibaca dengan **retrieval
k-NN** di atas ruang embedding yang dibentuk oleh ArcFace dan supervised contrastive loss.

Tiga keputusan utamanya, semuanya diturunkan dari analisis data:

1. **Preprocessing yang menormalkan polaritas dan memotong ke tinta.** Citra dibuat
   selalu tinta gelap di atas latar terang, kontras dinormalkan, bingkai hitam sisa rotasi
   dibuang, lalu dipotong ke wilayah bertinta.
2. **Augmentasi yang meniru kondisi test.** Rotasi, pewarnaan kertas dan tinta, blur,
   pikselasi, oklusi, bilah hitam, dan kompresi JPEG — setiap parameternya diturunkan dari
   perbandingan statistik train versus test.
3. **ArcFace + SupCon, dibaca lewat k-NN.** Macro-F1 menimbang semua kelas sama rata,
   sementara kelas terkecil (pegon) hanya punya 309 sampel. Head linear membutuhkan batas
   keputusan yang baik untuk kelas kecil; retrieval hanya membutuhkan klaster yang rapat.
   ArcFace dan SupCon membentuk klaster itu, k-NN membacanya.

**Catatan kepatuhan aturan.** Himpunan referensi k-NN hanya berisi data train berlabel.
Tidak ada informasi yang mengalir antar sampel test, sehingga ini klasifikasi supervised
induktif biasa — bukan label propagation, pseudo-labeling, atau semi-supervised, yang
dilarang Rule 7. Tidak ada data eksternal dan tidak ada metadata berkas yang dipakai
sebagai fitur: model hanya menerima piksel.

## 1. Persiapan dan konfigurasi

`ROOT` menunjuk folder yang memuat `data/train.csv`, `data/test.csv`, dan folder
`images/`. `EPOCHS` dikunci di muka karena training memakai seluruh data tanpa holdout,
sehingga tidak ada dasar untuk early stopping maupun pemilihan epoch.

In [ ]:
import gc, io, json, math, os, random, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image, ImageDraw, ImageFilter, ImageOps
from scipy import ndimage as ndi

import timm
from timm.optim import param_groups_layer_decay


def find_root():
    for c in [Path.cwd(), Path.cwd().parent, Path(r"E:/datsci/infest_usk"), Path("/kaggle/input")]:
        if (c / "data/train.csv").exists():
            return c
    raise FileNotFoundError("data/train.csv tidak ditemukan - set ROOT manual")


ROOT = find_root()
DATA = ROOT / "data"
IMG_DIR = next(p for p in [DATA / "images", ROOT / "images"] if (p / "train").exists())

SEED = 42
MODEL_NAME = "vit_large_patch16_dinov3.lvd1689m"
H, W = 160, 640                 # kanvas: tinggi tetap, lebar panjang untuk baris aksara
EPOCHS, WARMUP_EPOCHS = 10, 1
MICRO_BATCH, ACCUM = 16, 2      # batch efektif 32
EVAL_BATCH = 32
BASE_LR, HEAD_LR = 4e-5, 3e-4
LAYER_DECAY, WEIGHT_DECAY = 0.85, 0.05
LABEL_SMOOTHING = 0.1
NONLINE_BOOST = 1.5             # citra non line-like lebih sering muncul di test
RAW_MAX_SIDE = 1600
ARC_SCALE, ARC_MARGIN = 30.0, 0.20
SUPCON_W, SUPCON_T = 0.5, 0.1
KNN_K, KNN_TEMP = 5, 0.03
NUM_WORKERS = 0                 # >0 hanya jika lingkungan mendukung worker DataLoader
LABELS = ["bali", "jawa", "jawi", "lampung", "lontara", "pegon", "sunda"]
L2I = {l: i for i, l in enumerate(LABELS)}

os.environ["PYTHONHASHSEED"] = str(SEED)
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
print(dict(root=str(ROOT), device=str(DEVICE), amp=str(AMP), epochs=EPOCHS,
           torch=torch.__version__, timm=timm.__version__))

## 2. Preprocessing

Urutannya penting: `corrupt()` dijalankan **sebelum** `preprocess()` pada jalur training,
karena korupsi yang ditiru adalah kondisi citra test sebelum normalisasi. Dengan begitu
jaringan melihat citra train yang melewati normalisasi yang sama persis dengan citra test.

`remove_black_fill` membuang bidang hitam pekat yang menempel di tepi — sisa rotasi atau
bilah hitam — sebelum uji polaritas, karena bidang hitam besar bisa membalik seluruh citra
menjadi putih-di-atas-hitam. `crop_to_ink` memotong ke wilayah bertinta dengan penjaga agar
halaman renggang tidak runtuh menjadi satu coretan.

In [ ]:
"""Preprocessing + augmentation for INFEST aksara classification. Import this from the training notebook.

Order matters:   raw RGB --(train only) corrupt()--> preprocess() --> to_canvas()
corrupt() BEFORE preprocess() because the test corruptions were applied to raw images; the network must
see train images that went through exactly the same normalisation the corrupted test images go through.

Every number below comes from eda/outputs (see eda/README.md):
  rotation +-15 deg          07: test |skew| p95=10, p99=13; 30% of test lines >=3 deg (train 4%)
  tint p=.45                 03: bg_sat>12 in 44% test vs 3% train
  blur/pixelate p=.55        03: sharp<0.6 in 69% test vs 12% train
  occluder p=.12, dashes .15 03 flat_gray 6.6% test (detector under-counts, seen in 02 zoom)
  black bar p=.07            03: black border 7.3% test vs 1.2% train
  jpeg p=.3                  01: JPEG 21% test vs 4.6% train
  canvas H=160 fixed height  07: letterbox 160x640 leaves bali/lontara at ~2 patches of ink
"""
import io
import math
import random

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageOps
from scipy import ndimage as ndi


# ----------------------------------------------------------------------------- load
def load_rgb(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im)
        if im.mode in ("RGBA", "LA", "P", "PA"):
            im = im.convert("RGBA")
            im = Image.alpha_composite(Image.new("RGBA", im.size, "white"), im)
        return im.convert("RGB")


# ----------------------------------------------------------------------------- deterministic (train + test)
def _otsu(g):
    hist = np.bincount(g.ravel(), minlength=256).astype(float)
    p = hist / hist.sum(); w = np.cumsum(p); mu = np.cumsum(p * np.arange(256))
    return int(np.argmax((mu[-1] * w - mu) ** 2 / (w * (1 - w) + 1e-12)))


def remove_black_fill(g, dark=40, min_contact=0.3, min_solidity=0.9, max_holes=0.03):
    """Solid black regions glued to the border (shift / rotate fill, black bars) -> white.
    A border-touching pure-black component is fill when it has no holes (a real dark background has
    light text inside it) AND either (a) runs along >=min_contact of a side or (b) is near-convex
    (rotation corner triangle). Glyph-sized images (both sides <64px) only lose full-side bars.
    Only the thick core (survives 2px erosion) is removed, so thin strokes glued to the fill stay."""
    blk = g < dark
    if not blk.any():
        return g
    lab, n = ndi.label(blk)
    h, w = g.shape
    border = np.zeros_like(blk); border[0] = border[-1] = border[:, 0] = border[:, -1] = True
    tiny = max(h, w) < 64  # glyph-sized crops (e.g. a 20x20 bold numeral): only full-side bars count
    contact = 0.9 if tiny else min_contact
    objs = ndi.find_objects(lab)
    kill = set()
    for i in set(np.unique(lab[border & blk]).tolist()) - {0}:
        sl = objs[i - 1]
        comp = lab[sl] == i
        area = comp.sum()
        filled = ndi.binary_fill_holes(comp).sum()
        if (filled - area) / filled > max_holes or np.median(g[sl][comp]) > 15:
            continue
        run = max((lab[0] == i).sum() / w, (lab[-1] == i).sum() / w, (lab[:, 0] == i).sum() / h, (lab[:, -1] == i).sum() / h)
        if run >= contact:
            kill.add(i); continue
        if tiny or area < 0.005 * g.size:
            continue
        ys, xs = np.nonzero(comp)
        try:
            from scipy.spatial import ConvexHull
            hull = ConvexHull(np.c_[np.r_[ys, ys + 1, ys, ys + 1], np.r_[xs, xs, xs + 1, xs + 1]]).volume
        except Exception:
            continue
        if area / hull >= min_solidity:
            kill.add(i)
    if not kill:
        return g
    mask = np.isin(lab, list(kill))
    mask &= ndi.binary_dilation(ndi.binary_erosion(mask, iterations=2, border_value=1), iterations=3)
    out = g.copy()
    out[mask] = 255
    return out


def preprocess(rgb):
    """RGB PIL -> L PIL: gray, black fill removed, dark-ink-on-white, contrast normalised, cropped to ink.
    Fill removal runs BEFORE the polarity test: black rotation corners on a thin line can cover >50% of
    the frame and would otherwise flip the whole image to white-on-black (seen in debug_black_rotation.png)."""
    g = remove_black_fill(np.asarray(rgb.convert("L")))
    t = _otsu(g)
    if (g > t).mean() < 0.5:  # background is the majority class; if it is the dark side, invert
        g = 255 - g
    t = _otsu(g)
    ink, bg = g[g <= t], g[g > t]
    if ink.size and bg.size:
        lo, hi = np.percentile(ink, 10), np.median(bg)
        if hi - lo >= 20:  # skip blank / single-tone images
            g = np.clip((g.astype(np.float32) - lo) * 255.0 / (hi - lo), 0, 255).astype(np.uint8)
    return Image.fromarray(crop_to_ink(g))


def crop_to_ink(g, thr=128, trim=0.005, margin=0.08):
    ink = g < thr
    if ink.sum() < 20:
        return g
    def span(profile):
        c = np.cumsum(profile) / profile.sum()
        return int(np.searchsorted(c, trim)), int(np.searchsorted(c, 1 - trim))
    y0, y1 = span(ink.sum(1)); x0, x1 = span(ink.sum(0))
    if ink[y0:y1 + 1, x0:x1 + 1].mean() > 0.6:  # crop would be one solid blob (a black dash on a sparse page) -> keep all
        return g
    m = int(round(margin * max(y1 - y0, 8)))
    return g[max(0, y0 - m):y1 + m + 1, max(0, x0 - m):x1 + m + 1]


def to_canvas(img, H=160, W=640, train=False, rng=random):
    """Line-like crops (w/h>=2): resize to fixed height H, then random window (train) or tiles (eval).
    Others (posters, pages, square patches): letterbox. Returns list of HxW L images (len 1 when train)."""
    w, h = img.size
    if w / h >= 2:
        nw = max(1, round(w * H / h))
        img = img.resize((nw, H), Image.Resampling.BICUBIC if nw > w else Image.Resampling.LANCZOS)
        if nw <= W:
            c = Image.new("L", (W, H), 255); c.paste(img, ((W - nw) // 2, 0)); return [c]
        if train:
            x = rng.randint(0, nw - W); return [img.crop((x, 0, x + W, H))]
        n = math.ceil((nw - W) / (W // 2)) + 1
        xs = np.linspace(0, nw - W, n).round().astype(int)
        return [img.crop((x, 0, x + W, H)) for x in xs]
    s = min(H / h, W / w)
    img = img.resize((max(1, round(w * s)), max(1, round(h * s))), Image.Resampling.BICUBIC if s > 1 else Image.Resampling.LANCZOS)
    c = Image.new("L", (W, H), 255); c.paste(img, ((W - img.width) // 2, (H - img.height) // 2)); return [c]


# ----------------------------------------------------------------------------- train-only test-like corruption
PARCHMENT = [(236, 224, 196), (222, 205, 170), (245, 238, 214), (210, 196, 160), (196, 186, 160), (250, 246, 230), (170, 170, 170), (230, 230, 230)]
INKS = [(20, 20, 20), (90, 20, 30), (30, 70, 40), (40, 40, 90), (70, 50, 30), (60, 30, 80)]


def rotate(img, angle, fill="edge"):
    """RandomRotation that mimics the test set: frame size kept, corners filled by edge-replicate / white / black."""
    if fill == "edge":
        a = np.asarray(img); pad = max(a.shape[:2]) // 2
        big = Image.fromarray(np.pad(a, ((pad, pad), (pad, pad), (0, 0)), mode="edge"))
        big = big.rotate(angle, resample=Image.Resampling.BICUBIC)
        return big.crop((pad, pad, pad + img.width, pad + img.height))
    color = (255, 255, 255) if fill == "white" else (0, 0, 0)
    return img.rotate(angle, resample=Image.Resampling.BICUBIC, fillcolor=color)


def _ink_mask(img):
    g = np.asarray(img.convert("L")); return g < min(_otsu(g), 128)


def ink_retained(img, angle):
    """Share of the ORIGINAL ink still inside the frame after rotation (fill can't fake it)."""
    m = Image.fromarray(_ink_mask(img).astype(np.uint8) * 255)
    return (np.asarray(m.rotate(angle, resample=Image.Resampling.NEAREST, fillcolor=0)) > 0).sum() / max((np.asarray(m) > 0).sum(), 1)


def corrupt(img, rng=random):
    """RGB PIL -> RGB PIL with test-like corruptions. Rotation angle is re-drawn while <60% of the ink stays in frame."""
    w, h = img.size
    if rng.random() < 0.5:
        for _ in range(4):
            angle = rng.uniform(-15, 15)
            if ink_retained(img, angle) >= 0.6:
                img = rotate(img, angle, rng.choice(["edge"] * 5 + ["white"] * 2 + ["black"])); break
    if rng.random() < 0.25 and w > 60:  # partial crop along the line
        cw, ch = int(w * rng.uniform(0.6, 1.0)), int(h * rng.uniform(0.85, 1.0))
        x, y = rng.randint(0, w - cw), rng.randint(0, h - ch)
        img = img.crop((x, y, x + cw, y + ch)); w, h = img.size
    if rng.random() < 0.45:  # tint: re-colour ink and paper
        g = np.asarray(img.convert("L"), dtype=np.float32)[..., None] / 255
        bg = np.array(rng.choice(PARCHMENT), dtype=np.float32) * rng.uniform(0.85, 1.05)
        ink = np.array(rng.choice(INKS), dtype=np.float32) + rng.uniform(0, 60)
        img = Image.fromarray(np.clip(ink * (1 - g) + bg * g, 0, 255).astype(np.uint8))
    if rng.random() < 0.45:
        if rng.random() < 0.5:
            img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.2) * max(h, 32) / 64))
        else:
            f = rng.uniform(0.35, 0.8)
            small = img.resize((max(4, int(w * f)), max(4, int(h * f))), Image.Resampling.BILINEAR)
            img = small.resize((w, h), rng.choice([Image.Resampling.NEAREST, Image.Resampling.BILINEAR, Image.Resampling.BICUBIC]))
    if rng.random() < 0.08:  # semi-transparent gray polygons
        over = Image.new("RGBA", img.size, (0, 0, 0, 0)); d = ImageDraw.Draw(over)
        for _ in range(rng.randint(1, 3)):
            v = rng.randint(60, 150)
            d.polygon([(rng.uniform(-.1, 1.1) * w, rng.uniform(-.1, 1.1) * h) for _ in range(3)], fill=(v, v, v, rng.randint(120, 230)))
        img = Image.alpha_composite(img.convert("RGBA"), over).convert("RGB")
    if rng.random() < 0.15:  # small black dashes / cutout
        d = ImageDraw.Draw(img)
        for _ in range(rng.randint(1, 4)):
            rw, rh = rng.uniform(.03, .1) * w, rng.uniform(.03, .1) * h + 1
            x, y = rng.uniform(0, w - rw), rng.uniform(0, h - rh)
            d.rectangle([x, y, x + rw, y + rh], fill=(0, 0, 0))
    if rng.random() < 0.07:  # black bar from a vertical shift
        a = np.asarray(img).copy(); k = int(h * rng.uniform(0.05, 0.25)) + 1
        if rng.random() < 0.5:
            a[:-k] = a[k:]; a[-k:] = 0
        else:
            a[k:] = a[:-k]; a[:k] = 0
        img = Image.fromarray(a)
    if rng.random() < 0.15:
        a = np.asarray(img, dtype=np.float32) + np.random.default_rng(rng.randint(0, 2**31)).normal(0, rng.uniform(5, 25), (h, w, 1))
        img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    if rng.random() < 0.3:
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=rng.randint(20, 75)); buf.seek(0)
        img = Image.open(buf).convert("RGB")
    return img


def train_view(path, rng=random, H=160, W=640):
    return to_canvas(preprocess(corrupt(load_rgb(path), rng)), H, W, train=True, rng=rng)[0]


def eval_views(path, H=160, W=640):
    return to_canvas(preprocess(load_rgb(path)), H, W, train=False)

# ---- pemeriksaan mandiri
_img = Image.new("RGB", (400, 60), "white")
ImageDraw.Draw(_img).text((10, 20), "abc def ghi jkl", fill="black")
_p = preprocess(_img)
assert _p.mode == "L" and _p.width < 400 and np.asarray(_p).min() == 0
assert np.asarray(preprocess(ImageOps.invert(_img))).mean() > 128, "putih-di-atas-hitam harus dibalik"
_b = np.asarray(_img).copy(); _b[-15:] = 0
_g = remove_black_fill(np.asarray(Image.fromarray(_b).convert("L")))
assert (_g[-15:] == 255).all() and (_g < 128).sum() > 0, "bilah dibuang, teks dipertahankan"
_v = to_canvas(Image.new("L", (3000, 80), 255), H, W, train=False)
assert all(x.size == (W, H) for x in _v) and len(_v) > 1
_r = random.Random(0)
for _ in range(20):
    assert corrupt(_img, _r).mode == "RGB"
print("preprocessing self-check OK")

## 3. Memuat data

Seluruh citra dibaca sekali, diperkecil bila sisi terpanjangnya melebihi batas, lalu
disimpan dalam bentuk hasil `preprocess`. Dari situ dibentuk "ubin": satu citra bisa
menghasilkan lebih dari satu ubin bila ia berupa baris panjang, dan prediksinya nanti
dirata-rata atas seluruh ubin miliknya.

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
assert set(train.label) == set(LABELS) and train.image_id.is_unique and test.image_id.is_unique
assert not set(train.image_id) & set(test.image_id), "ID train dan test bertumpang tindih"
train["path"] = [str(IMG_DIR / "train" / i) for i in train.image_id]
test["path"] = [str(IMG_DIR / "test" / i) for i in test.image_id]
y_all = train.label.map(L2I).values
print(f"train={len(train)} (SELURUH data, tanpa holdout)  test={len(test)}")
print(train.label.value_counts().reindex(LABELS).to_string())


def load_cached(path):
    img = load_rgb(path)
    if max(img.size) > RAW_MAX_SIDE:
        img.thumbnail((RAW_MAX_SIDE, RAW_MAX_SIDE), Image.Resampling.LANCZOS)
    return np.asarray(preprocess(img))


t0 = time.time()
train_pp = [load_cached(p) for p in train.path]
test_pp = [load_cached(p) for p in test.path]
print(f"cache {time.time() - t0:.0f}s")

nonline = np.array([p.shape[1] / p.shape[0] < 2 for p in train_pp])
print(f"share citra non line-like di train: {nonline.mean():.1%}")


def build_tiles(pps):
    tiles, owner = [], []
    for k, pp in enumerate(pps):
        for t in to_canvas(Image.fromarray(pp), H, W, train=False):
            tiles.append(np.asarray(t, dtype=np.uint8)); owner.append(k)
    return np.stack(tiles), np.array(owner)


ref_tiles, ref_owner = build_tiles(train_pp)
test_tiles, test_owner = build_tiles(test_pp)
del train_pp, test_pp
gc.collect()
print(f"ubin train={len(ref_tiles)}  test={len(test_tiles)}")

Sebelum melanjutkan, kita lihat contoh citra hasil preprocessing dan hasil augmentasi
training, untuk memastikan aksaranya masih terbaca dan tidak rusak berlebihan.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(LABELS), 3, figsize=(18, 2.2 * len(LABELS)))
for r, lab in enumerate(LABELS):
    j = int(train.index[train.label == lab][0])
    raw = load_rgb(train.path[j])
    if max(raw.size) > RAW_MAX_SIDE:
        raw.thumbnail((RAW_MAX_SIDE, RAW_MAX_SIDE), Image.Resampling.LANCZOS)
    rng = random.Random(f"demo-{r}")
    axes[r][0].imshow(raw); axes[r][0].set_title(f"{lab}: mentah", fontsize=9)
    axes[r][1].imshow(to_canvas(preprocess(raw), H, W, train=False)[0], cmap="gray", vmin=0, vmax=255)
    axes[r][1].set_title("kanvas evaluasi", fontsize=9)
    axes[r][2].imshow(to_canvas(preprocess(corrupt(raw, rng)), H, W, train=True, rng=rng)[0],
                      cmap="gray", vmin=0, vmax=255)
    axes[r][2].set_title("tampilan training (korupsi + preprocess)", fontsize=9)
    for a in axes[r]: a.axis("off")
plt.tight_layout(); plt.show(); plt.close(fig)

## 4. Dataset training

Setiap pengambilan sampel menjalankan `corrupt` lalu `preprocess`, dengan seed yang
ditentukan oleh (SEED, epoch, indeks gambar) sehingga hasilnya dapat direproduksi berapa pun
jumlah worker yang dipakai.

In [ ]:
class AksaraDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths, self.labels = list(paths), np.asarray(labels)
        self.epoch = 0

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        rng = random.Random(f"{SEED}-{self.epoch}-{i}")
        img = load_rgb(self.paths[i])
        if max(img.size) > RAW_MAX_SIDE:
            img.thumbnail((RAW_MAX_SIDE, RAW_MAX_SIDE), Image.Resampling.LANCZOS)
        v = to_canvas(preprocess(corrupt(img, rng)), H, W, train=True, rng=rng)[0]
        return torch.from_numpy(np.asarray(v, np.uint8).copy()).unsqueeze(0), int(self.labels[i])


MEAN_G = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD_G = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def gpu_norm(x):
    """uint8 (B,1,H,W) -> float ternormalisasi (B,3,H,W)."""
    return (x.float().div_(255).expand(-1, 3, -1, -1) - MEAN_G) / STD_G


dataset = AksaraDataset(train.path.values, y_all)
print(f"dataset siap: {len(dataset)} sampel")

## 5. Model: DINOv3 ViT-L/16 dengan kepala ArcFace

Kepala ArcFace memberi margin sudut antar kelas, sehingga embedding satu kelas terkumpul
pada arah yang sama. Struktur itulah yang nanti dibaca oleh k-NN.

Learning rate memakai *layer-wise decay*: blok yang lebih dekat ke masukan dilatih dengan
LR lebih kecil agar representasi pretrained tidak rusak, sementara kepala klasifikasi
dilatih jauh lebih cepat.

In [ ]:
class ArcFaceHead(nn.Module):
    """Margin sudut: menarik satu kelas ke satu arah dan memberi jarak ke kelas lain."""

    def __init__(self, dim, classes, scale, margin):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(classes, dim))
        nn.init.xavier_uniform_(self.weight)
        self.scale, self.margin = scale, margin

    def forward(self, z, labels=None):
        cos = F.linear(F.normalize(z), F.normalize(self.weight))
        if labels is None:
            return cos * self.scale
        sin = torch.sqrt((1.0 - cos.square()).clamp_min(1e-7))
        phi = cos * math.cos(self.margin) - sin * math.sin(self.margin)
        oh = F.one_hot(labels, num_classes=cos.shape[1]).to(cos.dtype)
        return (oh * phi + (1.0 - oh) * cos) * self.scale


class ArcModel(nn.Module):
    def __init__(self, backbone, dim, n_cls):
        super().__init__()
        self.backbone = backbone
        self.head = ArcFaceHead(dim, n_cls, ARC_SCALE, ARC_MARGIN)

    def embed(self, x):
        return self.backbone(x)

    def forward(self, x, labels=None):
        return self.head(self.embed(x), labels)


def supcon_loss(z, y, temp):
    """Supervised contrastive: menarik sampel sekelas saling mendekat. Hanya label train."""
    z = F.normalize(z, dim=1)
    sim = z @ z.T / temp
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    pos = y[:, None].eq(y[None, :]) & ~eye
    if not pos.any():
        return z.new_tensor(0.0)
    sim = sim.masked_fill(eye, -1e4)
    lp = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    keep = pos.any(dim=1)
    return -(lp * pos).sum(1)[keep].div(pos.sum(1).clamp_min(1)[keep]).mean()


backbone = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0,
                             img_size=(H, W), global_pool="token")
model = ArcModel(backbone, backbone.num_features, len(LABELS)).to(DEVICE)

head_ids = {id(q) for q in model.head.parameters()}
groups = param_groups_layer_decay(backbone, weight_decay=WEIGHT_DECAY, layer_decay=LAYER_DECAY,
                                  no_weight_decay_list=backbone.no_weight_decay())
for g in groups:
    g["params"] = [q for q in g["params"] if id(q) not in head_ids]
groups = [g for g in groups if g["params"]]
groups.append({"params": list(model.head.parameters()),
               "weight_decay": WEIGHT_DECAY, "lr_scale": HEAD_LR / BASE_LR})
for g in groups:
    g["base_lr"] = BASE_LR * g.get("lr_scale", 1.0); g["lr"] = g["base_lr"]
assert sum(q.numel() for g in groups for q in g["params"]) == sum(q.numel() for q in model.parameters()), \
    "ada parameter yang tidak masuk optimizer"
optimizer = torch.optim.AdamW(groups, lr=BASE_LR)

counts = np.bincount(y_all, minlength=len(LABELS))
class_w = 1 / np.sqrt(counts); class_w = class_w / class_w.mean()
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_w, dtype=torch.float32, device=DEVICE),
                                label_smoothing=LABEL_SMOOTHING)
print(f"{MODEL_NAME}: {sum(q.numel() for q in model.parameters()) / 1e6:.1f}M parameter")
print("bobot kelas:", dict(zip(LABELS, class_w.round(3))))

## 6. Training

Sampling diberi bobot: citra non line-like lebih sering muncul di test daripada di train,
sehingga porsinya dinaikkan saat training. Learning rate menaik linear selama warm-up lalu
meluruh mengikuti cosine. Tidak ada early stopping karena tidak ada holdout.

In [ ]:
sample_w = torch.tensor(np.where(nonline, NONLINE_BOOST, 1.0), dtype=torch.double)
steps_per_epoch = (len(dataset) // MICRO_BATCH) // ACCUM
total_steps, warm_steps = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch


def lr_factor(step):
    if step < warm_steps:
        return (step + 1) / warm_steps
    p = min(1.0, (step - warm_steps) / max(1, total_steps - warm_steps))
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * p))


history, step = [], 0
t_start = time.time()
for epoch in range(EPOCHS):
    dataset.epoch = epoch
    sampler = WeightedRandomSampler(sample_w, num_samples=len(dataset), replacement=True,
                                    generator=torch.Generator().manual_seed(SEED * 1000 + epoch))
    loader = DataLoader(dataset, batch_size=MICRO_BATCH, sampler=sampler,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    model.train(); optimizer.zero_grad(set_to_none=True)
    t0, ce_sum, sup_sum, seen = time.time(), 0.0, 0.0, 0
    for it, (x, y) in enumerate(loader, 1):
        x = gpu_norm(x.to(DEVICE, non_blocking=True)); y = y.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, dtype=AMP, enabled=DEVICE.type == "cuda"):
            z = model.embed(x)
            ce = criterion(model.head(z, y).float(), y)
            sup = supcon_loss(z.float(), y, SUPCON_T)
            loss = ce + SUPCON_W * sup
        assert torch.isfinite(loss), f"loss tidak finite di epoch {epoch} iterasi {it}"
        (loss / ACCUM).backward()
        if it % ACCUM == 0:
            for g in optimizer.param_groups:
                g["lr"] = g["base_lr"] * lr_factor(step)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); optimizer.zero_grad(set_to_none=True); step += 1
        ce_sum += float(ce.detach()) * len(y); sup_sum += float(sup) * len(y); seen += len(y)
    rec = dict(epoch=epoch + 1, ce=ce_sum / seen, supcon=sup_sum / seen,
               seconds=round(time.time() - t0))
    history.append(rec); print(json.dumps(rec), flush=True)
print(f"training selesai dalam {(time.time() - t_start) / 60:.1f} menit")
pd.DataFrame(history)

## 7. Inferensi

Dua keluaran dihitung dari **model yang sama**:

- **head** — logits ArcFace langsung.
- **k-NN** — setiap citra test dibandingkan dengan seluruh embedding train berlabel, lalu
  diberi label berdasarkan tetangga terdekatnya dengan bobot cosine bertemperatur.

Nilai `k` dan temperature ditetapkan di muka. Himpunan referensi hanya berisi data train
berlabel, sehingga ini klasifikasi supervised induktif, bukan label propagation.

In [ ]:
@torch.inference_mode()
def pooled(tiles, owner, n, want):
    """Rata-rata per gambar atas seluruh ubin miliknya. want='logits' atau 'embed'."""
    model.eval()
    dim = len(LABELS) if want == "logits" else model.head.weight.shape[1]
    out = torch.zeros(n, dim, dtype=torch.float64)
    cnt = torch.zeros(n, 1, dtype=torch.float64)
    for s in range(0, len(tiles), EVAL_BATCH):
        x = gpu_norm(torch.from_numpy(tiles[s:s + EVAL_BATCH]).to(DEVICE).unsqueeze(1))
        with torch.autocast(DEVICE.type, dtype=AMP, enabled=DEVICE.type == "cuda"):
            z = model.embed(x)
            v = model.head(z, None) if want == "logits" else F.normalize(z.float(), dim=1)
        o = torch.from_numpy(owner[s:s + EVAL_BATCH]).long()
        out.index_add_(0, o, v.float().cpu().double())
        cnt.index_add_(0, o, torch.ones(len(o), 1, dtype=torch.float64))
    assert (cnt > 0).all(), "ada gambar tanpa ubin"
    return (out / cnt).numpy()


def weighted_knn(ref_x, ref_y, q_x, k, temperature):
    """k-NN berbobot cosine. Referensi HANYA data train berlabel (induktif)."""
    ref_x = ref_x / np.maximum(np.linalg.norm(ref_x, axis=1, keepdims=True), 1e-12)
    q_x = q_x / np.maximum(np.linalg.norm(q_x, axis=1, keepdims=True), 1e-12)
    sim = q_x @ ref_x.T
    k = min(k, len(ref_x))
    idx = np.argpartition(-sim, k - 1, axis=1)[:, :k]
    ss = np.take_along_axis(sim, idx, axis=1)
    lab = ref_y[idx]
    w = np.exp((ss - ss.max(axis=1, keepdims=True)) / max(temperature, 1e-6))
    probs = np.zeros((len(q_x), len(LABELS)), dtype="float32")
    for c in range(len(LABELS)):
        probs[:, c] = np.where(lab == c, w, 0).sum(axis=1)
    return probs / np.maximum(probs.sum(1, keepdims=True), 1e-12)


t0 = time.time()
test_logits = pooled(test_tiles, test_owner, len(test), "logits")
emb_ref = pooled(ref_tiles, ref_owner, len(train), "embed")
emb_test = pooled(test_tiles, test_owner, len(test), "embed")
print(f"inferensi {time.time() - t0:.0f}s")

knn_probs = weighted_knn(emb_ref, y_all, emb_test, KNN_K, KNN_TEMP)
pred_head = [LABELS[i] for i in test_logits.argmax(1)]
pred_knn = [LABELS[i] for i in knn_probs.argmax(1)]
print(f"kesepakatan head dan k-NN: {np.mean(np.array(pred_head) == np.array(pred_knn)):.3f}")

## 8. Submission

Berkas submission dibentuk langsung dari keluaran model; tidak ada pengisian manual.
Sebagai pemeriksaan akhir, distribusi kelas hasil prediksi dibandingkan dengan distribusi
kelas pada data train.

In [ ]:
submission = pd.DataFrame({"image_id": test.image_id, "label": pred_knn})
sample = pd.read_csv(DATA / "sample_submission.csv")
assert len(submission) == len(sample), "jumlah baris tidak cocok dengan sample_submission"
assert set(submission.image_id) == set(sample.image_id), "image_id tidak cocok"
assert submission.image_id.is_unique and not submission.isna().any().any()
assert set(submission.label) <= set(LABELS), "ada label di luar daftar kelas"
submission.to_csv("submission.csv", index=False)
print(f"submission.csv ditulis: {len(submission)} baris")

dist = pd.concat([
    pd.Series(pred_knn).value_counts(normalize=True).rename("prediksi_test"),
    train.label.value_counts(normalize=True).rename("prior_train")], axis=1).reindex(LABELS).round(4)
print(dist.to_string())
print()
print(submission.head().to_string(index=False))

## 9. Catatan penutup

Seluruh proses dijalankan hanya dengan data yang disediakan panitia. Tidak ada data
eksternal, tidak ada pseudo-labeling maupun label propagation, dan tidak ada metadata
berkas yang dipakai sebagai fitur klasifikasi — model hanya menerima tensor piksel.

Keputusan desain yang paling menentukan:

1. **Normalisasi polaritas dan pemotongan ke tinta**, sehingga citra dengan tinta terang
   di atas latar gelap tidak menjadi kelas tersendiri bagi model.
2. **Augmentasi yang meniru kondisi test**, dengan parameter yang diturunkan dari
   perbandingan statistik train versus test.
3. **ArcFace dan SupCon dibaca lewat k-NN**, yang menyasar kelas terkecil: macro-F1
   menimbang setiap kelas sama rata, sedangkan pegon hanya punya 309 sampel.